In [1]:
import numpy as np
import os
from pathlib import Path
import pandas as pd

from astropy.io import fits

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']
plt.style.use('dark_background')
cmap = sns.color_palette('mako', as_cmap=True)

### Fisher matrix

$$F_{\alpha\beta} = \frac{\partial \mathbf{O}^{T}}{\partial \theta_{\alpha}} \, \mathbf{C}^{-1} \, \frac{\partial \mathbf{O}}{\partial \theta_{\beta}}$$


If there's changes in the hod parameters:
$$\theta = \{\omega_b,\omega_{\mathrm{cdm}},n_s,\sigma_8, \theta_{\mathrm{HOD},1}, \theta_{\mathrm{HOD},2}, ..., \theta_{\mathrm{HOD},i}, \}$$

In [2]:
base = '/pscratch/sd/n/ntbfin/emulator/hods/z0.5/yuan23_prior'
!ls $base/c000_ph000/seed0/hod000.fits

/pscratch/sd/n/ntbfin/emulator/hods/z0.5/yuan23_prior/c000_ph000/seed0/hod000.fits


In [27]:
!ls /pscratch/sd/n/ntbfin/emulator/hods/v1.3/z0.5/yuan23_prior

c000_ph000  c109_ph000	c124_ph000  c142_ph000	c157_ph000  c172_ph000
c001_ph000  c110_ph000	c125_ph000  c143_ph000	c158_ph000  c173_ph000
c002_ph000  c111_ph000	c126_ph000  c144_ph000	c159_ph000  c174_ph000
c003_ph000  c112_ph000	c130_ph000  c145_ph000	c160_ph000  c175_ph000
c004_ph000  c113_ph000	c131_ph000  c146_ph000	c161_ph000  c176_ph000
c013_ph000  c114_ph000	c132_ph000  c147_ph000	c162_ph000  c177_ph000
c100_ph000  c115_ph000	c133_ph000  c148_ph000	c163_ph000  c178_ph000
c101_ph000  c116_ph000	c134_ph000  c149_ph000	c164_ph000  c179_ph000
c102_ph000  c117_ph000	c135_ph000  c150_ph000	c165_ph000  c180_ph000
c103_ph000  c118_ph000	c136_ph000  c151_ph000	c166_ph000  c181_ph000
c104_ph000  c119_ph000	c137_ph000  c152_ph000	c167_ph000
c105_ph000  c120_ph000	c138_ph000  c153_ph000	c168_ph000
c106_ph000  c121_ph000	c139_ph000  c154_ph000	c169_ph000
c107_ph000  c122_ph000	c140_ph000  c155_ph000	c170_ph000
c108_ph000  c123_ph000	c141_ph000  c156_ph000	c171_ph000


In [14]:
! find $base/c000_ph000/seed0 -maxdepth 1 -type f | wc -l

500


In [17]:
def save_hod_params(folder, output_csv, ext=1):
    folder = Path(folder)
    rows = []
    params = ['GAL_TYPE', 'Q_PAR', 'Q_PERP',
              'LOGM_CUT', 'LOGM1', 'SIGMA', 'ALPHA', 'KAPPA',
              'ALPHA_C', 'ALPHA_S',
              'S', 'S_V', 'S_P', 'S_R',
              'ACENT', 'ASAT', 'BCENT', 'BSAT', 'IC']

    for path in sorted(folder.glob('hod*.fits')):
        with fits.open(path) as hdul:
            header = hdul[ext].header
            
        row = {'filename': path.name, 'hod': path.stem}
        for p in params:
            row[p] = header.get(p, None)
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df

In [20]:
# df_hods = save_hod_params(f'{base}/c000_ph000/seed0', './data/params_hods_c000_ph000_seed0.csv')
df_hods = pd.read_csv('./data/params_hods_c000_ph000_seed0.csv')
df_hods.columns

Index(['filename', 'hod', 'GAL_TYPE', 'Q_PAR', 'Q_PERP', 'LOGM_CUT', 'LOGM1',
       'SIGMA', 'ALPHA', 'KAPPA', 'ALPHA_C', 'ALPHA_S', 'S', 'S_V', 'S_P',
       'S_R', 'ACENT', 'ASAT', 'BCENT', 'BSAT', 'IC'],
      dtype='object')

Now $F$ would be:

$$\theta_{\mathrm{cosmo}} = \{\omega_b, \omega_{\mathrm{cdm}}, n_s, \sigma_8\}$$

$$\theta_{\mathrm{full}} = \{\theta_{\mathrm{cosmo}}, \theta_{\mathrm{HOD}}\}$$

$$F = \begin{pmatrix}
F_{\mathrm{cosmo},\mathrm{cosmo}} &
F_{\mathrm{cosmo},\mathrm{HOD}} \\
F_{\mathrm{HOD},\mathrm{cosmo}} &
F_{\mathrm{HOD},\mathrm{HOD}}
\end{pmatrix}$$

In [28]:
# LOGM_CUT,LOGM1,SIGMA,ALPHA,KAPPA